# Resampling


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from starwinds_readplt.dataset import Dataset

from batcamp import Octree
from batcamp import OctreeInterpolator


## Build Dataset, Octree, and Interpolator

The file below is a Cartesian 3D sample. We build the tree once and reuse it for all queries.


In [2]:
import pooch

files = pooch.retrieve(
    url='https://zenodo.org/records/7110555/files/run-Sun-G2211.tar.gz',
    known_hash='c31a32aab08cc20d5b643bba734fd7220e6b369e691f55f88a3a08cc5b2a2136',
    progressbar=False,
    processor=pooch.Untar(members=['run-Sun-G2211/IH/IO2/3d__var_4_n00005000.plt']),
)
data_path = files[0]
data_path


/Users/dagfev/miniconda3/envs/batcamp/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'/Users/dagfev/Library/Caches/pooch/8c62a596be9356ac52956bdee5198bec-run-Sun-G2211.tar.gz.untar/run-Sun-G2211/IH/IO2/3d__var_4_n00005000.plt'

In [ ]:
ds = Dataset.from_file(str(data_path))

rho_name = next(name for name in ds.variables if name.startswith('Rho '))
tree = Octree.from_dataset(ds, tree_coord='xyz')
interp = OctreeInterpolator(ds, [rho_name], tree=tree)

xyz = np.asarray(ds.points[:, :3], dtype=float)
xyz_min = xyz.min(axis=0)
xyz_max = xyz.max(axis=0)
xyz_span = xyz_max - xyz_min
center = 0.5 * (xyz_min + xyz_max)

print(ds)
print('rho field:', rho_name)
print(tree)
print('xyz bounds min:', xyz_min)
print('xyz bounds max:', xyz_max)


Title:     'BATSRUS: 3D Data, 2011/02/16 17:26:00.000'
Zone:      '3D   N=0005000'
Variables: 24
Shape:     (4139545, 24)
Variables: ['X [R]', 'Y [R]', 'Z [R]', 'Rho [g/cm^3]', 'U_x [km/s]', 'U_y [km/s]', 'U_z [km/s]', 'ti [K]', 'te [K]', 'B_x [Gauss]', 'B_y [Gauss]', 'B_z [Gauss]', 'I01 [erg/cm^3]', 'I02 [erg/cm^3]', 'P [dyne/cm^2]', 'pe [dyne/cm^2]', 'E [erg/cm^3]', 'ehot [erg/cm^3]', 'qrad J/m^3/s', 'qheat J/m^3/s', 'refl 1/s', 'J_x [`mA/m^2]', 'J_y [`mA/m^2]', 'J_z [`mA/m^2]'].
rho field: Rho [g/cm^3]
Octree (adaptive): tree_coord=xyz, finest_leaf_grid=(256, 256, 256), root_grid=(1, 1, 1), depth=8, full=True, levels=0..3; leaf_levels[L0:5824 (fine-equiv 2981888), L1:80640 (fine-equiv 5160960), L2:697856 (fine-equiv 5582848), L3:3051520 (fine-equiv 3051520)]
xyz bounds min: [-450. -450. -450.]
xyz bounds max: [450. 450. 450.]


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


## Inspect Tree and Lookup

This section comes from the old Cartesian read notebook.


In [4]:
print(tree)
print('levels:', tree.levels)
print('is_uniform:', tree.is_uniform)
centers = tree.lookup.cell_centers
center = centers[len(centers) // 2]
hit = tree.lookup_point(center, coord='xyz')
print('center:', center)
print('cell_id:', None if hit is None else hit.cell_id)
print('level:', None if hit is None else hit.level)


Octree (adaptive): tree_coord=xyz, finest_leaf_grid=(256, 256, 256), root_grid=(1, 1, 1), depth=8, full=True, levels=0..3; leaf_levels[L0:5824 (fine-equiv 2981888), L1:80640 (fine-equiv 5160960), L2:697856 (fine-equiv 5582848), L3:3051520 (fine-equiv 3051520)]
levels: (0, 1, 2, 3)
is_uniform: False


AttributeError: 'OctreeLookup' object has no attribute '_cell_centers'

## Resample to XY Plane (z=0)

Dense Cartesian slice from the old large-file notebook.


In [ ]:
centers = np.asarray(tree.lookup.cell_centers, dtype=float)
xmin, xmax = float(np.min(centers[:, 0])), float(np.max(centers[:, 0]))
ymin, ymax = float(np.min(centers[:, 1])), float(np.max(centers[:, 1]))

nx, ny = 300, 300
x = np.linspace(xmin, xmax, nx)
y = np.linspace(ymin, ymax, ny)
xx, yy = np.meshgrid(x, y, indexing='xy')
zz = np.zeros_like(xx)
q = np.column_stack((xx.ravel(), yy.ravel(), zz.ravel()))

rho_xy, cell_xy = interp(q, query_coord='xyz', return_cell_ids=True)
rho_xy = np.asarray(rho_xy, dtype=float).reshape(ny, nx)
cell_xy = np.asarray(cell_xy, dtype=int).reshape(ny, nx)
valid_xy = (cell_xy >= 0) & (rho_xy > 0.0)
log_rho_xy = np.full_like(rho_xy, np.nan, dtype=float)
log_rho_xy[valid_xy] = np.log10(rho_xy[valid_xy])
print('xy valid fraction:', float(valid_xy.mean()))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 7), constrained_layout=True)
im = ax.pcolormesh(xx, yy, log_rho_xy, shading='auto', cmap='viridis')
ax.set_xlabel('X [R]')
ax.set_ylabel('Y [R]')
ax.set_title(f'log10({rho_name}) on z=0')
ax.set_aspect('equal')
cb = fig.colorbar(im, ax=ax)
cb.set_label('log10 rho')
plt.show()


## Resample to an Inclined Plane

Resolution: `320 x 240` points (dense enough for smooth structure, still fast).


In [ ]:
# Plane basis: normal is intentionally not axis-aligned.
normal = np.array([0.45, -0.20, 1.00], dtype=float)
normal /= np.linalg.norm(normal)

ref = np.array([0.0, 0.0, 1.0], dtype=float)
if abs(float(np.dot(normal, ref))) > 0.95:
    ref = np.array([1.0, 0.0, 0.0], dtype=float)

e_u = np.cross(normal, ref)
e_u /= np.linalg.norm(e_u)
e_v = np.cross(normal, e_u)

offset = np.array([0.10 * xyz_span[0], -0.05 * xyz_span[1], 0.08 * xyz_span[2]], dtype=float)
plane_center = center + offset

nu, nv = 320, 240
half_width_u = 0.40 * float(np.linalg.norm(xyz_span))
half_width_v = 0.28 * float(np.linalg.norm(xyz_span))

u = np.linspace(-half_width_u, half_width_u, nu)
v = np.linspace(-half_width_v, half_width_v, nv)
uu, vv = np.meshgrid(u, v, indexing='xy')

plane_xyz = (
    plane_center[None, None, :]
    + uu[..., None] * e_u[None, None, :]
    + vv[..., None] * e_v[None, None, :]
).reshape(-1, 3)

rho_plane, cell_plane = interp(plane_xyz, query_coord='xyz', return_cell_ids=True)
rho_plane = np.asarray(rho_plane, dtype=float).reshape(nv, nu)
cell_plane = np.asarray(cell_plane, dtype=int).reshape(nv, nu)

valid_plane = (cell_plane >= 0) & (rho_plane > 0.0)
log_rho_plane = np.full_like(rho_plane, np.nan, dtype=float)
log_rho_plane[valid_plane] = np.log10(rho_plane[valid_plane])

print('plane valid fraction:', float(valid_plane.mean()))


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6), constrained_layout=True)
im = ax.pcolormesh(uu, vv, log_rho_plane, shading='auto', cmap='viridis')
ax.set_xlabel('u coordinate in plane [R]')
ax.set_ylabel('v coordinate in plane [R]')
ax.set_title('Inclined plane: log10(rho)')
cb = fig.colorbar(im, ax=ax)
cb.set_label('log10 rho')
plt.show()


## Resample to a Sphere

Resolution: `180 x 360` (`theta x phi`) points.


In [ ]:
radii = np.linalg.norm(xyz, axis=1)
r_sphere = float(np.nanpercentile(radii, 42.0))

n_theta, n_phi = 180, 360
theta = np.linspace(0.0, np.pi, n_theta)
phi = np.linspace(0.0, 2.0 * np.pi, n_phi, endpoint=False)
pp, tt = np.meshgrid(phi, theta, indexing='xy')

x_s = r_sphere * np.sin(tt) * np.cos(pp)
y_s = r_sphere * np.sin(tt) * np.sin(pp)
z_s = r_sphere * np.cos(tt)

sphere_xyz = np.column_stack((x_s.ravel(), y_s.ravel(), z_s.ravel()))
rho_sphere, cell_sphere = interp(sphere_xyz, query_coord='xyz', return_cell_ids=True)
rho_sphere = np.asarray(rho_sphere, dtype=float).reshape(n_theta, n_phi)
cell_sphere = np.asarray(cell_sphere, dtype=int).reshape(n_theta, n_phi)

valid_sphere = (cell_sphere >= 0) & (rho_sphere > 0.0)
log_rho_sphere = np.full_like(rho_sphere, np.nan, dtype=float)
log_rho_sphere[valid_sphere] = np.log10(rho_sphere[valid_sphere])

print('sphere radius [R]:', r_sphere)
print('sphere valid fraction:', float(valid_sphere.mean()))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8), constrained_layout=True)
im = ax.pcolormesh(np.degrees(pp), np.degrees(tt), log_rho_sphere, shading='auto', cmap='plasma')
ax.set_xlabel('azimuth phi [deg]')
ax.set_ylabel('polar theta [deg]')
ax.set_title('Sphere sample: log10(rho)')
cb = fig.colorbar(im, ax=ax)
cb.set_label('log10 rho')
plt.show()
